# Limpieza de Datos — Precios de Propiedades en CABA

A partir del dataset crudo de ZonaProp, limpiamos y transformamos
cada columna para dejarla lista para el análisis.

## Pasos
1. Convertir strings a números
2. Manejar nulos
3. Limpiar barrios
4. Detectar y eliminar outliers
5. Exportar dataset limpio

### 1. Imports y carga

In [23]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../data/raw/zonaprop_raw.csv")
print(f"Dataset crudo: {df_raw.shape}")
df = df_raw.copy()
df.head()

Dataset crudo: (38298, 7)


,precio,expensas,metros,ambientes,banos,direccion,barrio
0,USD 530.000,$ 770.000 Expensas,172 m² tot.,4 amb.,3 baños,Soldado De La Independencia 1200,"Las Cañitas, Palermo"
1,USD 170.000,NaN,73 m² tot.,4 amb.,1 baño,Malabia al 200,"Villa Crespo, Capital Federal"
2,USD 120.000,$ 260.000 Expensas,54 m² tot.,3 amb.,1 baño,Paraguay e/ Jean Jaures y Doctor Tomás Manuel ...,"Recoleta, Capital Federal"
3,USD 220.000,$ 380.073 Expensas,68 m² tot.,3 amb.,2 baños,PICO 1600. Entre Libertador y 11 De Septiembre,"Núñez, Capital Federal"
4,USD 2.250.000,$ 2.100.000 Expensas,399 m² tot.,4 amb.,3 baños,Av. Alvear 1500,"Recoleta, Capital Federal"


### 2. Limpieza

#### 2.1 Limpieza columna precios

Primero veo si todos los precios estan en el mismo formato "USD X.XXX"

In [24]:
raros = df["precio"][~df["precio"].str.startswith("USD")].value_counts()
print(f"Valores fuera del formato USD: {len(raros)} tipos distintos")
print(raros)

Valores fuera del formato USD: 4 tipos distintos
precio
Consultar precio    175
$ 265.000             1
$ 110.000             1
$ 142.000             1
Name: count, dtype: int64


Hay 175 propiedades sin precio y otras 3 con el precio es Pesos Argentinos.
Son solo 178 filas sobre 35k, menos del 0.5%, las elimino ya que no se pierde nada relevante

In [25]:
def limpiar_precio(valor):
    if pd.isna(valor):
        return None
    # Eliminar casos sin precio numérico en USD
    if not valor.startswith("USD"):
        return None
    try:
        valor = valor.replace("USD ", "").replace(".", "")
        return int(valor)
    except ValueError:
        return None

df["precio"] = df["precio"].apply(limpiar_precio)

# Eliminar filas sin precio
df = df.dropna(subset=["precio"])

print(f"Filas después de limpiar precio: {len(df)}")
print(f"Filas eliminadas: {len(df_raw) - len(df)}")
print(f"\nMin: USD {df['precio'].min():,}")
print(f"Max: USD {df['precio'].max():,}")
print(f"Media: USD {df['precio'].mean():,.0f}")

Filas después de limpiar precio: 38120
Filas eliminadas: 178

Min: USD 1.0
Max: USD 50,520,900.0
Media: USD 400,594


#### 2.2 Limpieza columna expensas

Primero veo si todas las exprensas estan en el mismo formato "$ XXX.XXX Expensas"

In [ ]:
# Ver valores que no siguen el formato esperado
raros_exp = df["expensas"][
    df["expensas"].notna() & 
    ~df["expensas"].str.startswith("$")
].value_counts()

print(f"Valores fuera del formato: {len(raros_exp)} tipos distintos")
print(raros_exp)